# Version 2 – Model Validation on Combined Data (V1 model, Jan+Feb 2021)

**Task:** Evaluate the V1 model (trained on January 2021 only) on the new combined dataset (Jan+Feb 2021).  
**Model:** Loaded from `models/model_pkl_v1` (Google Drive via DVC) — **no retraining**.  
**Data:** `data/green_tripdata_2021-01.parquet` + `data/green_tripdata_2021-02.parquet` concatenated.  
**Version:** Data and evaluation code changed; model and training code unchanged from V1.

In [1]:
import pickle
import sys
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from HW1.data_preprocessing import load_and_process, train_val_test_split

In [2]:
repo_root = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
data = pd.DataFrame()

january = load_and_process("data/green_tripdata_2021-01.parquet", from_dvc=True, repo=repo_root)
february = load_and_process("data/green_tripdata_2021-02.parquet", from_dvc=True, repo=repo_root)

data = pd.concat([january, february])
data.head()

,passenger_count,trip_distance,fare_amount,pickup_hour,pickup_day_of_week,tip_applied
0,1.0,1.01,5.5,0,4,0
1,1.0,2.53,10.0,0,4,1
2,1.0,1.12,6.0,0,4,1
3,1.0,1.99,8.0,23,3,0
7,6.0,0.45,3.5,0,4,1


In [6]:
print(data.shape)

(71261, 6)


In [3]:
model = pickle.load(open('models/model_pkl_v1', 'rb'))

In [4]:
# Random split by index: no train data in test set (no leakage from split)
train, val, test = train_val_test_split(data, random_state=42)

# Use only test set for evaluation — features and target from test only
FEATURE_COLUMNS = [
    "passenger_count",
    "trip_distance",
    "pickup_hour",
    "pickup_day_of_week",
    "fare_amount",
]
TARGET_COLUMN = "tip_applied"

X_test = test[FEATURE_COLUMNS]
y_test = test[TARGET_COLUMN]

In [5]:
y_test_pred = model.predict(X_test)

print("Test set metrics:")
print("F1-score:", f1_score(y_test, y_test_pred))
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test set metrics:
F1-score: 0.6008445945945946
Accuracy: 0.5579045837231057
Precision: 0.5322459973065988
Recall: 0.6897420981190615

Confusion matrix:
[[2407 3126]
 [1600 3557]]
